
# ESPAC Crop Direct Field Emissions (DFE)

This notebook computes crop direct field emissions from `outputs/CSVs/02_espac_crop_lci_table_filtered__summary_<strategy>.csv` and writes:

- `outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_<strategy>.csv`

It inherits the crop summary strategy from notebook 2 metadata when available, loads factors from `inputs/03-05_dfe_factors.yml`, and propagates uncertainty with the matching crop uncertainty CSV.


In [1]:
from pathlib import Path
import re
import json
import unicodedata
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown, HTML

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_filtered_export_summary.json'

if not LATEST_FILTERED_SUMMARY_META_PATH.exists():
    raise RuntimeError(
        f"Missing metadata file: {LATEST_FILTERED_SUMMARY_META_PATH}. "
        "Run notebook 2 and export filtered CSV for the selected strategy first."
    )

try:
    summary_meta = json.loads(LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
except Exception as exc:
    raise RuntimeError(f"Failed reading strategy metadata {LATEST_FILTERED_SUMMARY_META_PATH}: {exc}") from exc

SUMMARY_STRATEGY = str(summary_meta.get('summary_level', '')).strip().lower()
ALLOWED_SUMMARY_STRATEGIES = {
    'province', 'region', 'crop_national', 'cropping_system',
    'irrig_m3_class', 'farm_size_class', 'crop_group', 'crop_group_national',
}
if not SUMMARY_STRATEGY:
    raise RuntimeError(
        f"Strategy is missing in metadata file {LATEST_FILTERED_SUMMARY_META_PATH}. "
        "Set Summary in notebook 2 and export filtered CSV before running notebook 3."
    )
if SUMMARY_STRATEGY not in ALLOWED_SUMMARY_STRATEGIES:
    raise RuntimeError(
        f"Unsupported summary strategy '{SUMMARY_STRATEGY}' in {LATEST_FILTERED_SUMMARY_META_PATH}. "
        f"Allowed values: {sorted(ALLOWED_SUMMARY_STRATEGIES)}"
    )

SUMMARY_SOURCE = 'metadata'
SUMMARY_TOKEN = SUMMARY_STRATEGY

available_strategy_inputs = sorted((PROJECT_DIR / 'outputs/CSVs').glob('02_espac_crop_lci_table_filtered__summary_*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
preferred_input_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{SUMMARY_TOKEN}.csv'
preferred_input_unc_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{SUMMARY_TOKEN}_uncertainty.csv'

INPUT_CSV = preferred_input_csv if preferred_input_csv.exists() else next((p for p in available_strategy_inputs if p.stem.endswith(f'__summary_{SUMMARY_TOKEN}')), preferred_input_csv)
INPUT_UNCERTAINTY_CSV = preferred_input_unc_csv if preferred_input_unc_csv.exists() else INPUT_CSV.with_name(f'{INPUT_CSV.stem}_uncertainty.csv')
OUTPUT_CSV = PROJECT_DIR / f'outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_{SUMMARY_TOKEN}.csv'
OUTPUT_UNCERTAINTY_CSV = OUTPUT_CSV.with_name(f"{OUTPUT_CSV.stem}_uncertainty.csv")

FACTORS_YML = PROJECT_DIR / 'inputs/03-05_dfe_factors.yml'
FACTORS_XLSX = PROJECT_DIR / 'inputs/03_Factors.xlsx'
NUTRIENT_CONTENTS_YML = PROJECT_DIR / 'inputs/03_dfe_nutrient_contents.yml'
REGENERATE_YAML_FROM_XLSX = False

assert INPUT_CSV.exists(), (
    f'Missing inherited strategy input CSV: {INPUT_CSV}. '
    f'Latest metadata: {LATEST_FILTERED_SUMMARY_META_PATH if LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}. '
    f'Available summary files: {[p.name for p in available_strategy_inputs]}'
)
assert INPUT_UNCERTAINTY_CSV.exists(), (
    f'Missing inherited strategy uncertainty CSV: {INPUT_UNCERTAINTY_CSV}. '
    f'Expected from inherited summary strategy: {SUMMARY_STRATEGY}'
)

assert FACTORS_YML.exists() or FACTORS_XLSX.exists(), 'Need inputs/03-05_dfe_factors.yml or inputs/03_Factors.xlsx'
assert NUTRIENT_CONTENTS_YML.exists(), f'Missing nutrient-content YAML: {NUTRIENT_CONTENTS_YML}'

print(f'Inherited summary strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})')
display(HTML(
    "<div style='padding:6px 10px; border:1px solid #d9d9d9; border-radius:6px; background:#f7f7f7;'>"
    f"<b>Inherited strategy:</b> <code>{SUMMARY_STRATEGY}</code>"
    "</div>"
))
print(f'Metadata file: {LATEST_FILTERED_SUMMARY_META_PATH if LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}')
print(f'Using inherited strategy input CSV: {INPUT_CSV}')
print(f'Using inherited strategy uncertainty CSV: {INPUT_UNCERTAINTY_CSV}')


from scripts.pipeline_manifest import (
    new_run_id,
    make_snapshot_copy,
    build_manifest_record,
    append_manifest_record,
    crop_otros_metadata_from_db,
)


Inherited summary strategy: crop_national (source: metadata)


Metadata file: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\02_latest_filtered_export_summary.json
Using inherited strategy input CSV: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_crop_lci_table_filtered__summary_crop_national.csv
Using inherited strategy uncertainty CSV: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_crop_lci_table_filtered__summary_crop_national_uncertainty.csv


In [2]:
def export_factors_xlsx_to_yaml(xlsx_path: Path, yml_path: Path) -> None:
    import pandas as pd

    factors_df = pd.read_excel(xlsx_path, sheet_name='factors')
    refs_df = pd.read_excel(xlsx_path, sheet_name='references')
    factors_df = factors_df.rename(columns={c: str(c).strip() for c in factors_df.columns})

    factor_values = {}
    factor_details = {}
    for _, row in factors_df.iterrows():
        var = str(row.get('Var', '')).strip()
        if not var or var.lower() == 'nan':
            continue
        val = float(row.get('Val', 0) or 0)
        factor_values[var] = val
        detail = {'value': val}
        if 'Comment' in factors_df.columns and pd.notna(row.get('Comment')):
            detail['comment'] = str(row.get('Comment'))
        if 'Current value' in factors_df.columns and pd.notna(row.get('Current value')):
            detail['current_value'] = str(row.get('Current value'))
        factor_details[var] = detail

    if yml_path.exists():
        with yml_path.open('r', encoding='utf-8') as f:
            existing = yaml.safe_load(f) or {}
    else:
        existing = {}

    existing.setdefault('meta', {})
    existing['meta']['source_excel'] = str(xlsx_path).replace('\\', '/')
    existing['meta']['source_sheet'] = 'factors'
    existing['knime_factors'] = factor_values
    existing['knime_factor_details'] = factor_details
    existing['references_sheet_preview'] = refs_df.fillna('').astype(str).head(10).to_dict(orient='records')

    with yml_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(existing, f, sort_keys=False, allow_unicode=True)


if REGENERATE_YAML_FROM_XLSX:
    export_factors_xlsx_to_yaml(FACTORS_XLSX, FACTORS_YML)
    print(f'Regenerated {FACTORS_YML} from {FACTORS_XLSX}')

with FACTORS_YML.open('r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f) or {}

KNIME_FACTORS = CFG.get('knime_factors', {})
ASSUMPTIONS = CFG.get('assumptions', {})
print('Loaded factors:', len(KNIME_FACTORS), 'from', FACTORS_YML)

with NUTRIENT_CONTENTS_YML.open('r', encoding='utf-8') as f:
    NUTRIENT_CFG = yaml.safe_load(f) or {}

NUTRIENT_MAPS = (NUTRIENT_CFG.get('espac_default_mappings', {}) or {})
print('Loaded nutrient-content mappings from', NUTRIENT_CONTENTS_YML)


Loaded factors: 52 from C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03-05_dfe_factors.yml
Loaded nutrient-content mappings from C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03_dfe_nutrient_contents.yml


In [3]:
def _norm_text(x: str) -> str:
    s = str(x).strip().upper()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r'\s+', ' ', s)
    return s


def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')


def get_factor(name: str, default: float = 0.0) -> float:
    try:
        return float(KNIME_FACTORS.get(name, default))
    except Exception:
        return float(default)


def infer_crop_type(crop: str, category: str) -> str:
    txt = _norm_text(crop)
    groups = (ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('keyword_groups', {}) or {}
    for crop_type, keywords in groups.items():
        for kw in keywords or []:
            if _norm_text(kw) in txt:
                return crop_type
    fallback = ((ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('fallback_by_category', {}) or {})
    return str(fallback.get(str(category).strip().lower(), 'cereal'))


def ensure_numeric_columns(df: pd.DataFrame, columns) -> pd.DataFrame:
    out = df.copy()
    for c in columns:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = to_num(out[c]).fillna(0.0)
    return out


def apply_fraction_columns(df: pd.DataFrame, source_col: str, frac_cfg: Dict[str, float], prefix: str):
    amount = to_num(df.get(source_col, 0)).fillna(0.0)
    n = amount * float(frac_cfg.get('N', 0.0))
    p = amount * float(frac_cfg.get('P', 0.0))
    k = amount * float(frac_cfg.get('K', 0.0))
    return {
        f'{prefix}_N': n,
        f'{prefix}_P': p,
        f'{prefix}_K': k,
    }


def build_min_max_scenarios(df_mid: pd.DataFrame, df_unc: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_min = df_mid.copy()
    df_max = df_mid.copy()
    for c in df_mid.columns:
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'
        if cmin in df_unc.columns:
            df_min[c] = pd.to_numeric(df_unc[cmin], errors='coerce').fillna(pd.to_numeric(df_min[c], errors='coerce'))
        if cmax in df_unc.columns:
            df_max[c] = pd.to_numeric(df_unc[cmax], errors='coerce').fillna(pd.to_numeric(df_max[c], errors='coerce'))
    return df_min, df_max


def build_uncertainty_output(df_out: pd.DataFrame, df_unc_in: pd.DataFrame, df_min_out: pd.DataFrame, df_max_out: pd.DataFrame) -> pd.DataFrame:
    key_cols = [c for c in ['Region', 'Province', 'Crop', 'Category'] if c in df_out.columns]
    unc_base = df_out[key_cols].copy() if key_cols else pd.DataFrame(index=df_out.index)
    unc_data = {}
    sample_count = pd.to_numeric(df_out.get('count', np.nan), errors='coerce')

    for c in df_out.columns:
        if not pd.api.types.is_numeric_dtype(df_out[c]):
            continue
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'

        if cmin in df_unc_in.columns and cmax in df_unc_in.columns:
            lo = pd.to_numeric(df_unc_in[cmin], errors='coerce')
            hi = pd.to_numeric(df_unc_in[cmax], errors='coerce')
        elif cmin in df_out.columns and cmax in df_out.columns:
            # Preserve precomputed bounds for derived metrics such as SOC that
            # are encoded directly on the output table rather than via min/max
            # scenario recomputation.
            lo = pd.to_numeric(df_out[cmin], errors='coerce')
            hi = pd.to_numeric(df_out[cmax], errors='coerce')
        else:
            lo = pd.to_numeric(df_min_out.get(c, df_out[c]), errors='coerce')
            hi = pd.to_numeric(df_max_out.get(c, df_out[c]), errors='coerce')

        mid = pd.to_numeric(df_out[c], errors='coerce')
        lo2 = np.minimum(np.minimum(lo, hi), mid)
        hi2 = np.maximum(np.maximum(lo, hi), mid)

        multi_obs = sample_count > 1
        point_bounds = (lo2 == hi2)
        force_spread = multi_obs & point_bounds & mid.notna()
        span = np.maximum(np.abs(mid) * 0.01, 1e-12)
        lo2 = np.where(force_spread & (mid != 0), np.maximum(0.0, mid - span), lo2)
        hi2 = np.where(force_spread & (mid != 0), mid + span, hi2)
        lo2 = np.where(force_spread & (mid == 0), 0.0, lo2)
        hi2 = np.where(force_spread & (mid == 0), 1e-12, hi2)

        unc_data[cmin] = pd.Series(lo2, index=df_out.index).fillna(mid)
        unc_data[cmax] = pd.Series(hi2, index=df_out.index).fillna(mid)

    unc_values = pd.DataFrame(unc_data, index=df_out.index)
    return pd.concat([unc_base, unc_values], axis=1).copy()


def _mode_non_null(series: pd.Series):
    x = series.dropna().astype(str).str.strip()
    x = x[x != '']
    if x.empty:
        return np.nan
    return x.value_counts().index[0]


def _combine_strategy8_crop_group_national(df: pd.DataFrame, uncertainty: bool = False) -> pd.DataFrame:
    if SUMMARY_TOKEN != 'crop_group_national' or 'Crop_group' not in df.columns:
        return df

    rows = []
    for crop_group, group in df.groupby('Crop_group', sort=False, dropna=False):
        rec = {}
        crop_group_txt = '' if pd.isna(crop_group) else str(crop_group).strip()
        rec['Crop_group'] = crop_group_txt
        if 'Category' in df.columns:
            rec['Category'] = crop_group_txt

        for col in df.columns:
            if col in {'Category', 'Crop_group'}:
                continue
            if col in {'Region', 'Province', 'Crop'}:
                rec[col] = _mode_non_null(group[col])
                continue

            values = pd.to_numeric(group[col], errors='coerce') if pd.api.types.is_numeric_dtype(df[col]) else group[col]
            if col == 'count':
                rec[col] = pd.to_numeric(values, errors='coerce').fillna(0.0).sum()
                continue
            if col == 'Area_ha':
                rec[col] = pd.to_numeric(values, errors='coerce').fillna(0.0).sum()
                continue

            if pd.api.types.is_numeric_dtype(df[col]):
                series = pd.to_numeric(values, errors='coerce')
                if uncertainty and col.endswith('__minValue'):
                    rec[col] = float(series.min()) if series.notna().any() else np.nan
                elif uncertainty and col.endswith('__maxValue'):
                    rec[col] = float(series.max()) if series.notna().any() else np.nan
                else:
                    rec[col] = float(series.median()) if series.notna().any() else np.nan
            else:
                rec[col] = _mode_non_null(group[col])

        rows.append(rec)

    out = pd.DataFrame(rows)
    ordered_cols = [c for c in df.columns if c in out.columns]
    out = out[ordered_cols].sort_values(['Crop_group']).reset_index(drop=True)
    return out


def apply_strategy_specific_export_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if SUMMARY_TOKEN == 'crop_group_national':
        out = _combine_strategy8_crop_group_national(out, uncertainty=bool(any(str(c).endswith('__minValue') or str(c).endswith('__maxValue') for c in out.columns)))
        if 'Crop_group' in out.columns:
            out['Category'] = out['Crop_group'].fillna('').astype(str)
        elif 'Category' in out.columns:
            out['Crop_group'] = out['Category'].fillna('').astype(str)
    return out


In [4]:
def compute_dfe_columns(df_in: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df_in.copy()

    # Expected source columns (missing ones are injected as zeros).
    source_numeric_cols = [
        'Yield_kgha', 'Seed_kgha',
        'Urea_kgha', 'Limestone_kgha', 'NPK_kgha', 'MAP_kgha', 'UAN_kgha', 'AN_kgha', 'AP_kgha', 'AS_kgha', 'CAN_kgha',
        'Organic_estiercol_kgha', 'Organic_fermentado_kgha', 'Organic_liquido_kgha', 'Total_fert_org_kgha',
        'Straw_kgha', 'Fuel_ha',
    ]
    df = ensure_numeric_columns(df, source_numeric_cols)

    if 'Crop' not in df.columns:
        df['Crop'] = ''
    if 'Category' not in df.columns:
        df['Category'] = ''

    # Crop type for residue-N defaults (KNIME script uses Custom_classification_3).
    df['crop_type_dfe'] = [infer_crop_type(c, cat) for c, cat in zip(df['Crop'], df['Category'])]

    crop_res_cfg = ASSUMPTIONS.get('crop_residues', {}) or {}
    n_content_by_crop_type = crop_res_cfg.get('n_content_by_crop_type', {}) or {}
    export_straw_default = float(crop_res_cfg.get('default_export_straw_fraction', 0.0) or 0.0)
    infer_straw = bool(crop_res_cfg.get('infer_straw_from_yield_if_missing', False))
    straw_ratio_map = crop_res_cfg.get('straw_to_yield_ratio_by_crop_type', {}) or {}

    straw = df['Straw_kgha'].copy()
    if infer_straw:
        inferred = pd.Series([
            float(straw_ratio_map.get(ct, 0.0)) for ct in df['crop_type_dfe']
        ], index=df.index) * df['Yield_kgha']
        straw = straw.where(straw > 0, inferred)
    export_straw = pd.Series(export_straw_default, index=df.index, dtype='float64')
    n_content_res = pd.Series([
        float(n_content_by_crop_type.get(ct, n_content_by_crop_type.get('cereal', 0.28 / 100)))
        for ct in df['crop_type_dfe']
    ], index=df.index, dtype='float64')
    N_crop_res = straw * (1 - export_straw.clip(lower=0, upper=1)) * n_content_res

    # Reconstruct nutrient totals using nutrient-content YAML (preferred), with fallback to legacy assumptions.
    fert_fracs = (NUTRIENT_MAPS.get('mineral_fertilizer_nutrient_fractions', {}) or ASSUMPTIONS.get('fertilizer_nutrient_fractions', {}) or {})
    org_fracs = (NUTRIENT_MAPS.get('organic_nutrient_fractions', {}) or ASSUMPTIONS.get('organic_nutrient_fractions', {}) or {})

    mineral_cols = ['Urea_kgha', 'Limestone_kgha', 'NPK_kgha', 'MAP_kgha', 'UAN_kgha', 'AN_kgha', 'AP_kgha', 'AS_kgha', 'CAN_kgha']
    min_N = pd.Series(0.0, index=df.index)
    min_P = pd.Series(0.0, index=df.index)
    min_K = pd.Series(0.0, index=df.index)
    for col in mineral_cols:
        fr = fert_fracs.get(col, {}) or {}
        min_N += df[col] * float(fr.get('N', 0.0))
        min_P += df[col] * float(fr.get('P', 0.0))
        min_K += df[col] * float(fr.get('K', 0.0))

    org_N = pd.Series(0.0, index=df.index)
    org_P = pd.Series(0.0, index=df.index)
    org_K = pd.Series(0.0, index=df.index)
    org_P_solid = pd.Series(0.0, index=df.index)
    org_P_liquid = pd.Series(0.0, index=df.index)
    for col, cfg in org_fracs.items():
        if col not in df.columns:
            df[col] = 0.0
        amount = df[col]
        n_part = amount * float((cfg or {}).get('N', 0.0))
        p_part = amount * float((cfg or {}).get('P', 0.0))
        k_part = amount * float((cfg or {}).get('K', 0.0))
        org_N += n_part
        org_P += p_part
        org_K += k_part
        phase = str((cfg or {}).get('phase', '')).strip().lower()
        if phase == 'solid':
            org_P_solid += p_part
        elif phase == 'liquid':
            org_P_liquid += p_part

    # KNIME-aligned derived nutrient totals
    df['N_min'] = min_N
    df['P_min'] = min_P
    df['K_min'] = min_K
    df['N_orga'] = org_N
    df['P_orga'] = org_P
    df['K_orga'] = org_K
    df['N_tot'] = df['N_min'] + df['N_orga']
    df['P_tot'] = df['P_min'] + df['P_orga']
    df['K_tot'] = df['K_min'] + df['K_orga']

    # KNIME script factors (direct field emissions block)
    EF_NH3_u = get_factor('EF_NH3_u')
    CF_NH3_u = get_factor('EF_NH3_u')  # KNIME script uses EF_NH3_u here (likely typo); preserved intentionally.
    EF_NH3_m = get_factor('EF_NH3_m')
    CF_NH3_m = get_factor('CF_NH3_m')
    FracLEACH = get_factor('FracLEACH')
    FracGASF = get_factor('FracGASF')
    FracGASM = get_factor('FracGASM')
    EF_N2O_d = get_factor('EF_N2O_d')
    EF_N2O_a = get_factor('EF_N2O_a')
    EF_N2O_l = get_factor('EF_N2O_l')
    EF_NOx_effective = FracGASM  # KNIME script sets EF_NOx = FracGASM.
    EF_limestone = get_factor('EF_limestone')
    EF_urea = get_factor('EF_urea')
    RUSLE_R = get_factor('RUSLE_R')
    RUSLE_K = get_factor('RUSLE_K')
    RUSLE_LS = get_factor('RUSLE_LS')
    RUSLE_C = get_factor('RUSLE_C')
    RUSLE_P = get_factor('RUSLE_P')
    Psoil = get_factor('Psoil')
    e = get_factor('e')
    r = get_factor('r')
    t = get_factor('t')
    kl = get_factor('kl')
    kr = get_factor('kr')
    ks1 = get_factor('ks1')
    ks2 = get_factor('ks2')
    kn = get_factor('kn')
    kt = get_factor('kt')
    s = get_factor('s')
    kd = get_factor('kd')

    N_applied = df['N_tot']
    P_applied = df['P_tot']
    _ = P_applied  # retained for traceability to KNIME variable names

    # Direct emissions from fertilisers (KNIME formulas)
    df['kgNH3N_ha'] = ((df['N_min'] * EF_NH3_u * CF_NH3_u) + (df['N_orga'] * EF_NH3_m * CF_NH3_m)) * 14 / 17
    df['kgNOxN_ha'] = (N_applied - df['kgNH3N_ha']) * EF_NOx_effective * 14 / 46
    df['kgNO3N_ha'] = N_applied * FracLEACH
    df['kgN2ON_ha_direct'] = (N_applied + N_crop_res) * EF_N2O_d
    df['kgN2ON_ha_indirect'] = (df['N_min'] * FracGASF + df['N_orga'] * FracGASM) * EF_N2O_a + (df['kgNO3N_ha'] * EF_N2O_l)
    df['kgN2ON_ha'] = df['kgN2ON_ha_direct'] + df['kgN2ON_ha_indirect']
    df['kgCO2urea_ha'] = df['Urea_kgha'] * EF_urea
    df['kgCO2limestone_ha'] = df['Limestone_kgha'] * EF_limestone
    df['kgCO2_fert_ha'] = df['kgCO2urea_ha'] + df['kgCO2limestone_ha']

    soil_erosion_tha = RUSLE_R * RUSLE_K * RUSLE_LS * RUSLE_C * RUSLE_P
    df['soil_erosion_tha'] = soil_erosion_tha

    fm = df['P_min']
    fos = org_P_solid
    fol = org_P_liquid
    df['kgP_ha_erosion'] = soil_erosion_tha * Psoil * e * r
    df['kgP_ha_runoff'] = s * kr * ks1 * kn * kt * (1 + 0.2 / 80 * fm + 0.7 / 80 * fol + 0.4 / 80 * fos)
    df['kgP_ha_leaching'] = kl * ks2 * kn * (1 + 0.2 / 80 * fol)
    df['kgP_ha_drainage'] = df['kgP_ha_leaching'] * kd
    df['kgP_total'] = df['kgP_ha_erosion'] + df['kgP_ha_runoff'] + df['kgP_ha_leaching'] + df['kgP_ha_drainage']

    # Trace elements: compute row-level metals from organic fertiliser amounts when present,
    # using PRO-derived median metal contents from inputs/03_dfe_nutrient_contents.yml (mg/kg product, wet-matter basis).
    # Fallback to legacy constants from inputs/03-05_dfe_factors.yml for rows without organic inputs.
    trace_metals = ['Cd', 'Cu', 'Zn', 'Pb', 'Ni', 'Cr', 'Hg']
    organic_trace_cfg = (NUTRIENT_MAPS.get('organic_trace_metal_factors_mg_per_kg', {}) or {})
    organic_trace_basis_mass = pd.Series(0.0, index=df.index, dtype='float64')
    trace_from_organic = {m: pd.Series(0.0, index=df.index, dtype='float64') for m in trace_metals}

    for org_col, metal_map in organic_trace_cfg.items():
        if org_col not in df.columns:
            continue
        amount = to_num(df[org_col]).fillna(0.0)
        organic_trace_basis_mass = organic_trace_basis_mass + amount
        for metal in trace_metals:
            mg_per_kg = float((metal_map or {}).get(metal, 0.0) or 0.0)
            trace_from_organic[metal] = trace_from_organic[metal] + amount * mg_per_kg * 1e-6

    for metal in trace_metals:
        # Data-driven policy: emit trace metals only when organic fertiliser input is present.
        df[f'{metal}_kg_ha'] = np.where(organic_trace_basis_mass > 0, trace_from_organic[metal], 0.0)

    df['trace_metals_from_organic'] = organic_trace_basis_mass > 0

    # Soil organic carbon (SOC) sequestration for permanent systems (pastures and tree crops)
    soc_cfg = (ASSUMPTIONS.get('soc_sequestration', {}) or {})
    
    # Initialize SOC columns (0 for all before assignment)
    df['SOC_mean_rate_kgChayr'] = 0.0
    df['SOC_mean_rate_kgChayr__minValue'] = 0.0
    df['SOC_mean_rate_kgChayr__maxValue'] = 0.0
    df['SOC_source'] = ''

    # Normalize context fields so SOC assignment still works after aggregation levels change.
    category_lc = df.get('Category', pd.Series('', index=df.index)).astype(str).str.lower().str.strip()
    crop_group_lc = df.get('Crop_group', pd.Series('', index=df.index)).astype(str).str.lower().str.strip()

    # 1. Permanent pastures (Brachiaria, Panicum: South America-specific improved pasture rates)
    pasture_cfg = soc_cfg.get('pasture', {}) or {}
    pasture_mean = float(pasture_cfg.get('mean_kgChayr', 900.0))
    pasture_min = float(pasture_cfg.get('min_kgChayr', 300.0))
    pasture_max = float(pasture_cfg.get('max_kgChayr', 1500.0))
    is_pasture = (category_lc == 'cultivated_pasture') | (crop_group_lc == 'forages_pastures')
    df.loc[is_pasture, 'SOC_mean_rate_kgChayr'] = pasture_mean
    df.loc[is_pasture, 'SOC_mean_rate_kgChayr__minValue'] = pasture_min
    df.loc[is_pasture, 'SOC_mean_rate_kgChayr__maxValue'] = pasture_max
    df.loc[is_pasture, 'SOC_source'] = 'pasture_south_america'
    
    # 2. Permanent tree crops (coffee, cacao, bananas, etc.)
    # Tree crop rates vary by species; ranges from IPCC Tier 1 tropical agroforestry + Ecuador field studies
    import unicodedata

    def _normalize_crop_name(value):
        normalized = unicodedata.normalize('NFKD', str(value))
        stripped = ''.join(ch for ch in normalized if not unicodedata.combining(ch))
        return stripped.upper().strip()

    tree_cfg = soc_cfg.get('tree_crops', {}) or {}
    crop_name_upper = df['Crop'].apply(_normalize_crop_name)
    
    # Coffee: shade-grown agroforestry (1400 kg C/ha/yr mean) Ã¢â‚¬â€ Ecuador dominant practice
    coffee_mask = crop_name_upper.str.contains('CAFE|COFFEE', na=False, regex=True)
    if coffee_mask.any():
        coffee_params = tree_cfg.get('coffee', {}) or {}
        df.loc[coffee_mask, 'SOC_mean_rate_kgChayr'] = float(coffee_params.get('mean_kgChayr', 1400.0))
        df.loc[coffee_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(coffee_params.get('min_kgChayr', 600.0))
        df.loc[coffee_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(coffee_params.get('max_kgChayr', 2200.0))
        df.loc[coffee_mask, 'SOC_source'] = 'coffee_shade_agroforestry'
    
    # Cacao: shade-grown traditional system (1100 kg C/ha/yr mean) Ã¢â‚¬â€ dominant in ManabÃƒÂ­/Esmeraldas
    cacao_mask = crop_name_upper.str.contains('CACAO|COCOA', na=False, regex=True)
    if cacao_mask.any():
        cacao_params = tree_cfg.get('cacao', {}) or {}
        df.loc[cacao_mask, 'SOC_mean_rate_kgChayr'] = float(cacao_params.get('mean_kgChayr', 1100.0))
        df.loc[cacao_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(cacao_params.get('min_kgChayr', 400.0))
        df.loc[cacao_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(cacao_params.get('max_kgChayr', 1800.0))
        df.loc[cacao_mask, 'SOC_source'] = 'cacao_shade_traditional'
    
    # Bananas/Plantains: monoculture baseline lower (400 kg C/ha/yr) but can improve with mulching
    banana_mask = crop_name_upper.str.contains('BANANO|BANANA|PLATANO|ORITO', na=False, regex=True)
    if banana_mask.any():
        banana_params = tree_cfg.get('banana_plantain', {}) or {}
        df.loc[banana_mask, 'SOC_mean_rate_kgChayr'] = float(banana_params.get('mean_kgChayr', 400.0))
        df.loc[banana_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(banana_params.get('min_kgChayr', 100.0))
        df.loc[banana_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(banana_params.get('max_kgChayr', 800.0))
        df.loc[banana_mask, 'SOC_source'] = 'banana_commercial_monoculture'
    
    # Citrus/other fruits: 500 kg C/ha/yr baseline
    citrus_mask = crop_name_upper.str.contains('NARANJA|LIMON|MANDARINA|CITRICO|CITRUS', na=False, regex=True)
    if citrus_mask.any():
        citrus_params = tree_cfg.get('citrus', {}) or {}
        df.loc[citrus_mask, 'SOC_mean_rate_kgChayr'] = float(citrus_params.get('mean_kgChayr', 500.0))
        df.loc[citrus_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(citrus_params.get('min_kgChayr', 150.0))
        df.loc[citrus_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(citrus_params.get('max_kgChayr', 900.0))
        df.loc[citrus_mask, 'SOC_source'] = 'citrus_conventional'
    
    # Avocado: 850 kg C/ha/yr (highland preference in Ecuador)
    avocado_mask = crop_name_upper.str.contains('AGUACATE|AVOCADO', na=False, regex=True)
    if avocado_mask.any():
        avocado_params = tree_cfg.get('avocado', {}) or {}
        df.loc[avocado_mask, 'SOC_mean_rate_kgChayr'] = float(avocado_params.get('mean_kgChayr', 850.0))
        df.loc[avocado_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(avocado_params.get('min_kgChayr', 600.0))
        df.loc[avocado_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(avocado_params.get('max_kgChayr', 1100.0))
        df.loc[avocado_mask, 'SOC_source'] = 'avocado_highland'

    # Mango: FAO AGRIS orchard rate available
    mango_mask = crop_name_upper.str.contains('MANGO', na=False, regex=True)
    if mango_mask.any():
        mango_params = tree_cfg.get('mango', {}) or {}
        df.loc[mango_mask, 'SOC_mean_rate_kgChayr'] = float(mango_params.get('mean_kgChayr', 1530.0))
        df.loc[mango_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(mango_params.get('min_kgChayr', 1000.0))
        df.loc[mango_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(mango_params.get('max_kgChayr', 2000.0))
        df.loc[mango_mask, 'SOC_source'] = 'mango_fao_agris_orchard'

    # Passionfruit / maracuyÃƒÂ¡: FAO-backed agroforestry proxy
    passionfruit_mask = crop_name_upper.str.contains('MARACUYA|PASSION', na=False, regex=True)
    if passionfruit_mask.any():
        passionfruit_params = tree_cfg.get('passionfruit', {}) or {}
        df.loc[passionfruit_mask, 'SOC_mean_rate_kgChayr'] = float(passionfruit_params.get('mean_kgChayr', 210.0))
        df.loc[passionfruit_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(passionfruit_params.get('min_kgChayr', 30.0))
        df.loc[passionfruit_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(passionfruit_params.get('max_kgChayr', 450.0))
        df.loc[passionfruit_mask, 'SOC_source'] = 'agroforestry_proxy_fao_lac'

    # Pineapple: FAO evidence supports near-zero net SOC change
    pineapple_mask = crop_name_upper.str.contains('PINA', na=False, regex=True)
    if pineapple_mask.any():
        pineapple_params = tree_cfg.get('pineapple', {}) or {}
        df.loc[pineapple_mask, 'SOC_mean_rate_kgChayr'] = float(pineapple_params.get('mean_kgChayr', 0.0))
        df.loc[pineapple_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(pineapple_params.get('min_kgChayr', 0.0))
        df.loc[pineapple_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(pineapple_params.get('max_kgChayr', 100.0))
        df.loc[pineapple_mask, 'SOC_source'] = 'pineapple_fao_near_zero'

    # Tree tomato / tomate de arbol: FAO-backed agroforestry proxy
    tree_tomato_mask = crop_name_upper.str.contains('TOMATE DE ARBOL|SOLANUM BETACEUM', na=False, regex=True)
    if tree_tomato_mask.any():
        tree_tomato_params = tree_cfg.get('tree_tomato', {}) or {}
        df.loc[tree_tomato_mask, 'SOC_mean_rate_kgChayr'] = float(tree_tomato_params.get('mean_kgChayr', 210.0))
        df.loc[tree_tomato_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(tree_tomato_params.get('min_kgChayr', 30.0))
        df.loc[tree_tomato_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(tree_tomato_params.get('max_kgChayr', 450.0))
        df.loc[tree_tomato_mask, 'SOC_source'] = 'agroforestry_proxy_fao_lac'

    # African oil palm: FAO AGRIS managed-range placeholder
    oil_palm_mask = crop_name_upper.str.contains('PALMA AFRICANA|OIL PALM|ELAEIS', na=False, regex=True)
    if oil_palm_mask.any():
        oil_palm_params = tree_cfg.get('oil_palm', {}) or {}
        df.loc[oil_palm_mask, 'SOC_mean_rate_kgChayr'] = float(oil_palm_params.get('mean_kgChayr', 950.0))
        df.loc[oil_palm_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(oil_palm_params.get('min_kgChayr', 300.0))
        df.loc[oil_palm_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(oil_palm_params.get('max_kgChayr', 1900.0))
        df.loc[oil_palm_mask, 'SOC_source'] = 'oil_palm_fao_agris_managed'

    # Palmito / peach palm: FAO-backed agroforestry proxy
    peach_palm_mask = crop_name_upper.str.contains('PALMITO|PEACH PALM|BACTRIS', na=False, regex=True)
    if peach_palm_mask.any():
        peach_palm_params = tree_cfg.get('peach_palm', {}) or {}
        df.loc[peach_palm_mask, 'SOC_mean_rate_kgChayr'] = float(peach_palm_params.get('mean_kgChayr', 210.0))
        df.loc[peach_palm_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(peach_palm_params.get('min_kgChayr', 30.0))
        df.loc[peach_palm_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(peach_palm_params.get('max_kgChayr', 450.0))
        df.loc[peach_palm_mask, 'SOC_source'] = 'agroforestry_proxy_fao_lac'

    # Unspecified permanent perennials: FAO-backed agroforestry proxy
    other_permanent_mask = crop_name_upper.str.fullmatch('OTROS PERMANENTES', na=False)
    if other_permanent_mask.any():
        other_params = tree_cfg.get('other_permanents', {}) or {}
        df.loc[other_permanent_mask, 'SOC_mean_rate_kgChayr'] = float(other_params.get('mean_kgChayr', 210.0))
        df.loc[other_permanent_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(other_params.get('min_kgChayr', 30.0))
        df.loc[other_permanent_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(other_params.get('max_kgChayr', 450.0))
        df.loc[other_permanent_mask, 'SOC_source'] = 'agroforestry_proxy_fao_lac'

    # Sugarcane rows classified as permanent in ESPAC
    sugarcane_mask = crop_name_upper.str.contains('CANA DE AZUCAR|SUGARCANE', na=False, regex=True)
    if sugarcane_mask.any():
        sugarcane_params = tree_cfg.get('sugarcane', {}) or {}
        df.loc[sugarcane_mask, 'SOC_mean_rate_kgChayr'] = float(sugarcane_params.get('mean_kgChayr', 750.0))
        df.loc[sugarcane_mask, 'SOC_mean_rate_kgChayr__minValue'] = float(sugarcane_params.get('min_kgChayr', 0.0))
        df.loc[sugarcane_mask, 'SOC_mean_rate_kgChayr__maxValue'] = float(sugarcane_params.get('max_kgChayr', 830.0))
        df.loc[sugarcane_mask, 'SOC_source'] = 'sugarcane_fao_modelled'

    # 3. Aggregation fallback: when crop/category semantics are replaced by group summaries
    # (for example, Category='fruits' and Crop='(all crops in group)'), assign SOC from
    # crop-group defaults so perennial groups still carry SOC factors.
    group_soc_cfg = soc_cfg.get('crop_group_fallback', {}) or {}

    def _group_soc_params(group_key: str, default_mean: float, default_min: float, default_max: float):
        g = (group_soc_cfg.get(group_key, {}) or {})
        return (
            float(g.get('mean_kgChayr', default_mean)),
            float(g.get('min_kgChayr', default_min)),
            float(g.get('max_kgChayr', default_max)),
        )

    # Conservative defaults; can be overridden in inputs/03-05_dfe_factors.yml:soc_sequestration.crop_group_fallback
    fruit_mean, fruit_min, fruit_max = _group_soc_params('fruits', 500.0, 150.0, 900.0)
    ind_mean, ind_min, ind_max = _group_soc_params('industrial_cash', 750.0, 0.0, 1800.0)

    needs_soc_fallback = (df['SOC_mean_rate_kgChayr'] <= 0)
    fruits_group_mask = needs_soc_fallback & ((crop_group_lc == 'fruits') | (category_lc == 'fruits'))
    industrial_group_mask = needs_soc_fallback & ((crop_group_lc == 'industrial_cash') | (category_lc == 'industrial_cash'))

    if fruits_group_mask.any():
        df.loc[fruits_group_mask, 'SOC_mean_rate_kgChayr'] = fruit_mean
        df.loc[fruits_group_mask, 'SOC_mean_rate_kgChayr__minValue'] = fruit_min
        df.loc[fruits_group_mask, 'SOC_mean_rate_kgChayr__maxValue'] = fruit_max
        df.loc[fruits_group_mask, 'SOC_source'] = 'crop_group_fallback_fruits'

    if industrial_group_mask.any():
        df.loc[industrial_group_mask, 'SOC_mean_rate_kgChayr'] = ind_mean
        df.loc[industrial_group_mask, 'SOC_mean_rate_kgChayr__minValue'] = ind_min
        df.loc[industrial_group_mask, 'SOC_mean_rate_kgChayr__maxValue'] = ind_max
        df.loc[industrial_group_mask, 'SOC_source'] = 'crop_group_fallback_industrial_cash'

    # Helpful diagnostics on reconstructed inputs
    diagnostics = pd.DataFrame({
        'metric': [
            'rows',
            'rows_with_positive_N_tot',
            'rows_with_positive_P_tot',
            'rows_with_positive_seed',
            'rows_with_positive_dfe_N2O',
            'rows_with_soc_pastures',
            'rows_with_soc_coffee',
            'rows_with_soc_cacao',
            'rows_with_soc_banana_plantain',
            'rows_with_soc_citrus',
            'rows_with_soc_avocado',
            'rows_with_soc_sequestration_total',
        ],
        'value': [
            len(df),
            int((df['N_tot'] > 0).sum()),
            int((df['P_tot'] > 0).sum()),
            int((df['Seed_kgha'] > 0).sum()) if 'Seed_kgha' in df.columns else 0,
            int((df['kgN2ON_ha'] > 0).sum()),
            int(is_pasture.sum()),
            int(coffee_mask.sum()),
            int(cacao_mask.sum()),
            int(banana_mask.sum()),
            int(citrus_mask.sum()),
            int(avocado_mask.sum()),
            int((df['SOC_mean_rate_kgChayr'] > 0).sum()),
        ]
    })

    return df, diagnostics

### Uncertainty Processing (new)
This notebook now propagates uncertainty through DFE formulas.
- It reads `outputs/CSVs/02_espac_crop_lci_table_filtered__summary_<strategy>_uncertainty.csv` when available.
- It computes DFE with min-input and max-input scenarios, then writes bounds as `__minValue` / `__maxValue`.
- It exports `outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_<strategy>_uncertainty.csv` for XML generation.



In [5]:
df_in = pd.read_csv(INPUT_CSV)
print('Input shape:', df_in.shape)

if INPUT_UNCERTAINTY_CSV.exists():
    df_unc_in = pd.read_csv(INPUT_UNCERTAINTY_CSV)
    print('Input uncertainty shape:', df_unc_in.shape)
else:
    df_unc_in = pd.DataFrame(index=df_in.index)
    print(f'Input uncertainty CSV not found: {INPUT_UNCERTAINTY_CSV}. Using point values as min/max fallback.')

# Preserve original column order and append DFE columns at the end.
df_out, dfe_diag = compute_dfe_columns(df_in)

# Build min/max DFE scenarios from uncertainty input bounds.
df_min_in, df_max_in = build_min_max_scenarios(df_in, df_unc_in)
df_min_out, _ = compute_dfe_columns(df_min_in)
df_max_out, _ = compute_dfe_columns(df_max_in)

df_unc_out = build_uncertainty_output(df_out, df_unc_in, df_min_out, df_max_out)

# Drop helper source columns that were injected as zero placeholders only because they were absent in the input.
synthetic_source_helpers = ['Urea_kgha', 'Limestone_kgha', 'MAP_kgha', 'UAN_kgha', 'CAN_kgha', 'Straw_kgha']
for c in synthetic_source_helpers:
    if c not in df_in.columns and c in df_out.columns:
        df_out = df_out.drop(columns=[c])
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'
        df_unc_out = df_unc_out.drop(columns=[x for x in [cmin, cmax] if x in df_unc_out.columns], errors='ignore')

new_cols = [c for c in df_out.columns if c not in df_in.columns]
print('New columns added:', len(new_cols))
print(new_cols)

# Optional clean-up: reorder so original columns stay first.
df_out = df_out[list(df_in.columns) + new_cols]

# Export labels can intentionally differ from the internal classification used for DFE calculations.
df_out = apply_strategy_specific_export_labels(df_out)
df_unc_out = apply_strategy_specific_export_labels(df_unc_out)

# Write outputs
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved DFE-augmented table to: {OUTPUT_CSV}')
print('Output shape:', df_out.shape)

df_unc_out.to_csv(OUTPUT_UNCERTAINTY_CSV, index=False, encoding='utf-8-sig')
print(f'Saved DFE uncertainty table to: {OUTPUT_UNCERTAINTY_CSV}')

# Immutable run snapshots + manifest record
manifest_run_id = new_run_id('03_05_crops')
main_snapshot = make_snapshot_copy(OUTPUT_CSV, manifest_run_id)
unc_snapshot = make_snapshot_copy(OUTPUT_UNCERTAINTY_CSV, manifest_run_id)
summary_meta_for_manifest = summary_meta if isinstance(summary_meta, dict) else {}
otros_meta = {
    'otros_expansion_source': 'db',
    'subcrop_count_by_label': {
        'OTROS PERMANENTES': crop_otros_metadata_from_db(PROJECT_DIR, 'OTROS PERMANENTES'),
        'OTROS TRANSITORIOS': crop_otros_metadata_from_db(PROJECT_DIR, 'OTROS TRANSITORIOS'),
    },
}
manifest_record = build_manifest_record(
    run_id=manifest_run_id,
    domain='crops',
    summary_token=SUMMARY_TOKEN,
    pipeline_stage='03-05',
    source_main_csv=main_snapshot,
    source_unc_csv=unc_snapshot,
    main_df=df_out,
    unc_df=df_unc_out,
    filters_meta=summary_meta_for_manifest,
    upstream_run_id=str(summary_meta_for_manifest.get('run_id', '') or None),
    extra=otros_meta,
)
manifest_path = append_manifest_record(PROJECT_DIR, manifest_record)
print(f'Appended manifest record: {manifest_path} (run_id={manifest_run_id})')

# Quick summaries
num_show = [
    'N_tot', 'P_tot', 'K_tot',
    'kgNH3N_ha', 'kgNOxN_ha', 'kgNO3N_ha', 'kgN2ON_ha',
    'kgCO2_fert_ha', 'kgP_total'
]
num_show = [c for c in num_show if c in df_out.columns]
if num_show:
    preview_cols = [c for c in ['Crop', 'Crop_group', 'Category', 'Region', 'Province'] if c in df_out.columns]
    display(df_out[preview_cols + num_show].head(15))
    display(df_out[num_show].describe().T)

display(dfe_diag)


Input shape: (46, 39)
Input uncertainty shape: (46, 102)


New columns added: 37
['crop_type_dfe', 'N_min', 'P_min', 'K_min', 'N_orga', 'P_orga', 'K_orga', 'N_tot', 'P_tot', 'K_tot', 'kgNH3N_ha', 'kgNOxN_ha', 'kgNO3N_ha', 'kgN2ON_ha_direct', 'kgN2ON_ha_indirect', 'kgN2ON_ha', 'kgCO2urea_ha', 'kgCO2limestone_ha', 'kgCO2_fert_ha', 'soil_erosion_tha', 'kgP_ha_erosion', 'kgP_ha_runoff', 'kgP_ha_leaching', 'kgP_ha_drainage', 'kgP_total', 'Cd_kg_ha', 'Cu_kg_ha', 'Zn_kg_ha', 'Pb_kg_ha', 'Ni_kg_ha', 'Cr_kg_ha', 'Hg_kg_ha', 'trace_metals_from_organic', 'SOC_mean_rate_kgChayr', 'SOC_mean_rate_kgChayr__minValue', 'SOC_mean_rate_kgChayr__maxValue', 'SOC_source']
Saved DFE-augmented table to: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_crop_lci_table_filtered_dfe__summary_crop_national.csv
Output shape: (46, 76)
Saved DFE uncertainty table to: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_crop_lci_table_filtered_dfe__summary_crop_national_uncertainty.csv


Appended manifest record: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\pipeline_run_manifest.json (run_id=03_05_crops_20260507T111130Z)


,Crop,Category,Region,Province,N_tot,P_tot,K_tot,kgNH3N_ha,kgNOxN_ha,kgNO3N_ha,kgN2ON_ha,kgCO2_fert_ha,kgP_total
0,AGUACATE (FRUTA FRESCA),permanent,(all regions confounded),(all provinces confounded),60.064361,29.909737,57.185042,1.188391,3.404567,18.619952,0.537576,0.0,1.722203
1,ARROZ (EN CÁSCARA),transitory,(all regions confounded),(all provinces confounded),70.100000,29.457000,68.483250,1.386949,3.973407,21.731000,0.627395,0.0,1.722005
2,ARVEJA SECA (GRANO SECO),transitory,(all regions confounded),(all provinces confounded),45.321438,5.196127,10.921459,0.896698,2.568909,14.049646,0.405627,0.0,1.711391
3,ARVEJA TIERNA (VAINA),transitory,(all regions confounded),(all provinces confounded),82.049724,38.858986,26.818615,1.623378,4.650741,25.435415,0.734345,0.0,1.726118
4,BANANO (FRUTA FRESCA),permanent,(all regions confounded),(all provinces confounded),82.266422,28.146727,89.650800,1.627665,4.663024,25.502591,0.736284,0.0,1.721432
5,BRACHIARIA,cultivated_pasture,(all regions confounded),(all provinces confounded),23.625304,2.618400,54.786600,0.467434,1.339129,7.323844,0.211446,0.0,1.710263
6,BROCOLI,transitory,(all regions confounded),(all provinces confounded),125.191494,26.035035,102.415351,2.476951,7.096102,38.809363,1.120464,0.0,1.720508
7,CACAO (ALMENDRA SECA),permanent,(all regions confounded),(all provinces confounded),63.625708,24.547500,56.031750,1.258853,3.606431,19.723970,0.569450,0.0,1.719857
8,CAFÉ (GRANO ORO),permanent,(all regions confounded),(all provinces confounded),74.650000,40.257900,46.693125,1.476972,4.231310,23.141500,0.668118,0.0,1.726730
9,CAÑA DE AZÚCAR / AZÚCAR (TALLO FRESCO),permanent,(all regions confounded),(all provinces confounded),113.000000,28.829607,87.160500,2.235738,6.405064,35.030000,1.011350,0.0,1.721731


,count,mean,std,min,25%,50%,75%,max
N_tot,46.0,74.080986,43.635229,7.500000,49.150393,65.489043,81.375484,252.181535
P_tot,46.0,30.189205,36.685515,2.618400,12.952205,24.995826,33.857773,243.267628
K_tot,46.0,74.634756,101.256038,6.225750,33.670850,55.409175,77.004982,572.321080
kgNH3N_ha,46.0,1.509230,0.939730,0.148390,0.972455,1.295720,1.626593,4.989486
kgNOxN_ha,46.0,4.196541,2.470293,0.425115,2.785942,3.712049,4.612524,14.294149
kgNO3N_ha,46.0,22.965106,13.526921,2.325000,15.236622,20.301603,25.226400,78.176276
kgN2ON_ha,46.0,0.663965,0.391707,0.067125,0.439896,0.586127,0.728311,2.257025
kgCO2_fert_ha,46.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
kgP_total,46.0,1.722450,0.016093,1.710263,1.714784,1.720053,1.723930,1.815547


,metric,value
0,rows,46
1,rows_with_positive_N_tot,46
2,rows_with_positive_P_tot,46
3,rows_with_positive_seed,24
4,rows_with_positive_dfe_N2O,46
5,rows_with_soc_pastures,6
6,rows_with_soc_coffee,1
7,rows_with_soc_cacao,1
8,rows_with_soc_banana_plantain,3
9,rows_with_soc_citrus,2
